# Lab 02 · 임베딩 검색과 RAG를 처음부터 끝까지

**이 실습이 다루는 자료** ([필독 목록](../READING_LIST.md) · [실습 가이드](../PRACTICE.md))

| 절 | 직접 확인할 것 | 관련 자료 |
|---|---|---|
| 1 | 글자 겹침으로 찾는 희소 검색(TF-IDF) | 정보 검색의 고전적 기준선 |
| 2 | 의미로 찾는 밀집 검색(임베딩) | *Sentence-BERT* (2019), *Dense Passage Retrieval* (2020) |
| 3 | 검색 품질을 숫자로: Recall@k, MRR, 그리고 두 방법의 결합 | *MTEB* (2022) |
| 4 | 검색한 문서를 프롬프트에 넣어 답하게 하기 | Lewis et al. (2020) *RAG* |
| 5 | 문서 없이 답할 때와 비교해 **평가**하기 | *Your AI Product Needs Evals*, *Lost in the Middle* (2023) |

**소요 시간**: 60~90분 · **패키지**: numpy, scikit-learn, sentence-transformers, transformers, torch  
Colab: `!pip install -q sentence-transformers` (나머지는 기본 포함). 모델 두 개(임베딩 약 0.5GB, 생성 약 1GB)를 내려받습니다. CPU로 3~5분이면 끝납니다.

> 사용하는 모델: 임베딩 `intfloat/multilingual-e5-small` (MIT), 생성 `Qwen/Qwen2.5-0.5B-Instruct` (Apache-2.0). 둘 다 가중치가 공개된 작은 모델입니다.

In [1]:
import numpy as np, pandas as pd, warnings, time
warnings.filterwarnings("ignore")

## 0. 실습용 문서: 가상의 데이터팀 위키

RAG가 필요한 상황을 만들기 위해, **모델이 학습 중에 볼 수 없었던** 가상의 사내 규칙을 문서로 씁니다. 뒤쪽 4개는 일반 용어 설명(방해 문서 역할)입니다.

In [2]:
docs = [
 "실험 정책: 우리 팀의 A/B 테스트 기본 유의수준은 0.01이며 최소 실험 기간은 14일이다. 요일 효과를 고려해 연장은 7일 단위로만 한다.",
 "핵심 지표 정의: 핵심 지표(OEC)는 주간 구매 전환율이다. 분모는 주간 활성 사용자, 분자는 구매를 완료한 사용자이며 환불한 사용자는 분자에서 제외한다.",
 "데이터 분할 규칙: 모델 검증은 반드시 시간 기준으로 나눈다. 학습은 2025년 12월까지, 검증은 2026년 1월과 2월, 테스트는 2026년 3월 데이터를 쓴다.",
 "배포 절차: 모델 배포는 매주 화요일 오전 10시에만 한다. 금요일 배포는 금지한다. 배포 전에 그림자(shadow) 테스트를 3일 동안 진행한다.",
 "피처 관리: 피처는 사내 도구 Baguni에 등록한다. 피처 이름은 소문자 스네이크 케이스로 쓰고 접두사로 도메인 이름을 붙인다.",
 "장애 대응: 모델 성능 경보가 울리면 30분 안에 1차 대응을 한다. AUC가 기준보다 0.03 이상 떨어지면 이전 버전으로 되돌린다.",
 "추천 시스템 구조: 홈 화면 추천은 투타워 검색 모델로 후보 500개를 뽑고 LightGBM 랭커가 상위 20개를 고른다.",
 "수요 예측 기준: 수요 예측의 기준 모델은 계절성 나이브이며, 새 모델은 MASE가 0.9 미만일 때만 채택한다.",
 "LLM 사용 정책: 고객 문의 요약에 LLM을 쓸 때는 개인정보를 마스킹한 뒤 전송한다. 프롬프트와 응답 기록은 90일 동안 보관한다.",
 "LLM 평가 규칙: LLM 평가 결과는 반드시 표준오차와 함께 보고한다. 문항 수가 300개 미만인 벤치마크로는 모델의 우열을 결론짓지 않는다.",
 "SQL 작성 규칙: 쿼리는 CTE로 단계를 나누어 쓴다. SELECT * 는 금지한다. 날짜 필터는 파티션 컬럼 dt에 건다.",
 "논문 읽기 모임: 논문 읽기 모임은 격주 목요일 오후 4시에 연다. 발표자는 14개 질문 템플릿으로 요약을 준비한다.",
 "오류 코드 E-4012: 피처 저장소 동기화가 지연되었다는 뜻이다. 15분 뒤 자동으로 다시 시도하므로 기다린다.",
 "오류 코드 E-4021: 학습 작업의 메모리가 부족하다는 뜻이다. 배치 크기를 절반으로 줄여 다시 실행한다.",
 "용어 설명: p-value는 귀무가설이 참일 때 관측한 것만큼 극단적인 결과가 나올 확률이다. 가설이 참일 확률이 아니다.",
 "용어 설명: CUPED는 실험 전 데이터를 공변량으로 써서 지표의 분산을 줄이는 방법이다. 같은 표본으로 더 작은 효과를 검출할 수 있다.",
 "용어 설명: 과적합은 모델이 학습 데이터의 잡음까지 외워서 새로운 데이터에서 성능이 떨어지는 현상이다.",
 "용어 설명: 어텐션은 한 토큰을 표현할 때 다른 토큰들을 얼마나 참고할지 가중치를 계산해 섞는 연산이다.",
]
# (질문, 정답 문서 번호, 답에 꼭 들어가야 할 핵심 표현)
queries = [
 ("실험은 최소 며칠 동안 돌려야 하나요?", 0, "14"),
 ("금요일에 모델을 올려도 되나요?", 3, "금지"),
 ("성능이 얼마나 나빠지면 롤백합니까?", 5, "0.03"),
 ("새 예측 모델을 채택하는 조건은?", 7, "0.9"),
 ("고객 정보를 언어모델에 보낼 때 지켜야 할 것은?", 8, "마스킹"),
 ("문항이 200개인 시험으로 어느 모델이 더 나은지 결론 내려도 되나요?", 9, "300"),
 ("테스트 세트는 언제 데이터인가요?", 2, "3월"),
 ("쿼리에서 모든 컬럼을 한꺼번에 선택해도 되나요?", 10, "금지"),
 ("첫 화면 추천 후보는 몇 개를 뽑나요?", 6, "500"),
 ("전환율을 계산할 때 환불한 사람도 세나요?", 1, "제외"),
 ("스터디는 언제 하나요?", 11, "목요일"),
 ("분산을 줄여 실험 민감도를 높이는 방법은?", 15, "CUPED"),
 ("E-4021 이 뜨면 어떻게 하나요?", 13, "절반"),
 ("E-4012 는 무슨 뜻인가요?", 12, "동기화"),
]
print(len(docs), "documents,", len(queries), "queries")

18 documents, 14 queries


## 1. 희소 검색: 글자가 얼마나 겹치는가 (TF-IDF)

질문과 문서를 "어떤 글자 조각이 몇 번 나왔는가"의 벡터로 바꾸고 코사인 유사도로 순위를 매깁니다. 한국어는 조사·어미가 붙으므로 단어 대신 **글자 2~3개 조각(n-gram)** 을 씁니다.

✍️ **예측**: "스터디는 언제 하나요?" 라는 질문으로 "논문 읽기 모임은 격주 목요일…" 문서를 찾을 수 있을까요?

In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 3)).fit(docs)
doc_sparse = tfidf.transform(docs)

def rank_sparse(query):
    scores = (doc_sparse @ tfidf.transform([query]).T).toarray()[:, 0]
    return np.argsort(-scores)

for q in ["스터디는 언제 하나요?", "E-4021 이 뜨면 어떻게 하나요?"]:
    top = rank_sparse(q)[:2]
    print("Q:", q); [print("   ", i, docs[i][:48], "...") for i in top]

Q: 스터디는 언제 하나요?
    4 피처 관리: 피처는 사내 도구 Baguni에 등록한다. 피처 이름은 소문자 스네이크 케 ...
    10 SQL 작성 규칙: 쿼리는 CTE로 단계를 나누어 쓴다. SELECT * 는 금지한다. ...
Q: E-4021 이 뜨면 어떻게 하나요?
    13 오류 코드 E-4021: 학습 작업의 메모리가 부족하다는 뜻이다. 배치 크기를 절반으로 ...
    12 오류 코드 E-4012: 피처 저장소 동기화가 지연되었다는 뜻이다. 15분 뒤 자동으로 ...


"스터디"와 "논문 읽기 모임"은 **글자가 전혀 겹치지 않아서** 찾지 못합니다. 반대로 `E-4021` 같은 **정확한 코드**는 아주 잘 찾습니다.

## 2. 밀집 검색: 의미가 가까운가 (문장 임베딩)

임베딩 모델은 문장을 384차원 벡터로 바꿉니다. 뜻이 비슷하면 글자가 달라도 벡터가 가깝도록 학습되어 있습니다 (Sentence-BERT, DPR).  
e5 계열은 질문에는 `query: `, 문서에는 `passage: ` 를 앞에 붙이도록 학습됐습니다. **모델 카드를 읽지 않으면 놓치는 사용법**입니다.

In [4]:
from sentence_transformers import SentenceTransformer
t0 = time.time()
embedder = SentenceTransformer("intfloat/multilingual-e5-small", device="cpu")
doc_dense = embedder.encode(["passage: " + d for d in docs], normalize_embeddings=True)     # 길이 1로 정규화 → 내적 = 코사인 유사도
print("문서 임베딩 shape:", doc_dense.shape, f"({time.time() - t0:.0f}초)")

def rank_dense(query):
    q = embedder.encode(["query: " + query], normalize_embeddings=True)[0]
    return np.argsort(-(doc_dense @ q))

for q in ["스터디는 언제 하나요?", "E-4021 이 뜨면 어떻게 하나요?"]:
    top = rank_dense(q)[:2]
    print("Q:", q); [print("   ", i, docs[i][:48], "...") for i in top]

문서 임베딩 shape: (18, 384) (4초)
Q: 스터디는 언제 하나요?
    2 데이터 분할 규칙: 모델 검증은 반드시 시간 기준으로 나눈다. 학습은 2025년 12월 ...
    11 논문 읽기 모임: 논문 읽기 모임은 격주 목요일 오후 4시에 연다. 발표자는 14개 질 ...
Q: E-4021 이 뜨면 어떻게 하나요?
    13 오류 코드 E-4021: 학습 작업의 메모리가 부족하다는 뜻이다. 배치 크기를 절반으로 ...
    12 오류 코드 E-4012: 피처 저장소 동기화가 지연되었다는 뜻이다. 15분 뒤 자동으로 ...


밀집 검색은 글자가 전혀 겹치지 않는 "스터디" 질문에서도 정답 문서를 **2위**까지 끌어올립니다(희소 검색에서는 한참 아래). 1위가 아니라는 점도 눈여겨보세요. "언제"라는 말 때문에 날짜가 많이 나오는 문서가 더 가깝게 계산됐습니다. 임베딩 유사도는 "정답을 담고 있는가"가 아니라 "비슷한 내용인가"를 잽니다.

## 3. 검색 품질을 숫자로 재기

"잘 찾는 것 같다"로 끝내지 않고, 정답 문서가 몇 번째에 나왔는지로 지표를 계산합니다.

- **Recall@k**: 정답 문서가 상위 k개 안에 있는 질문의 비율
- **MRR**: 정답 순위의 역수(1위=1, 2위=0.5, …)의 평균

두 순위를 합치는 간단한 방법인 **RRF**(Reciprocal Rank Fusion)도 함께 봅니다: 각 방법에서의 순위 $r$ 에 대해 $1/(60+r)$ 을 더한 점수로 다시 정렬.

✍️ **예측**: 희소·밀집·결합 중 어느 것이 가장 좋을까요? 각 방법은 어떤 질문에서 틀릴까요?

In [5]:
def rank_hybrid(query, k=60):
    score = np.zeros(len(docs))
    for ranking in (rank_sparse(query), rank_dense(query)):
        for position, doc_id in enumerate(ranking): score[doc_id] += 1 / (k + position + 1)
    return np.argsort(-score)

def evaluate(rank_fn):
    ranks = np.array([int(np.where(rank_fn(q) == gold)[0][0]) + 1 for q, gold, _ in queries])
    return {"Recall@1": (ranks == 1).mean(), "Recall@3": (ranks <= 3).mean(), "MRR": (1 / ranks).mean()}, ranks

results, all_ranks = {}, {}
for name, fn in {"sparse (TF-IDF)": rank_sparse, "dense (e5)": rank_dense, "hybrid (RRF)": rank_hybrid}.items():
    results[name], all_ranks[name] = evaluate(fn)
print(pd.DataFrame(results).T.round(2), "\n")
print("정답 문서의 순위 (1이면 성공)")
print(pd.DataFrame(all_ranks, index=[q[:28] for q, _, _ in queries]).to_string())

                 Recall@1  Recall@3   MRR
sparse (TF-IDF)      0.86      0.93  0.89
dense (e5)           0.93      1.00  0.96
hybrid (RRF)         0.93      0.93  0.94 

정답 문서의 순위 (1이면 성공)
                              sparse (TF-IDF)  dense (e5)  hybrid (RRF)
실험은 최소 며칠 동안 돌려야 하나요?                       1           1             1
금요일에 모델을 올려도 되나요?                           1           1             1
성능이 얼마나 나빠지면 롤백합니까?                         3           1             1
새 예측 모델을 채택하는 조건은?                          1           1             1
고객 정보를 언어모델에 보낼 때 지켜야 할 것은?                 1           1             1
문항이 200개인 시험으로 어느 모델이 더 나은지                 1           1             1
테스트 세트는 언제 데이터인가요?                          1           1             1
쿼리에서 모든 컬럼을 한꺼번에 선택해도 되나요?                  1           1             1
첫 화면 추천 후보는 몇 개를 뽑나요?                       1           1             1
전환율을 계산할 때 환불한 사람도 세나요?                     1           1             1
스터디는 언제 하나요?       

표에서 확인할 것
- 희소 검색은 **표현이 바뀐 질문**에서 순위가 밀립니다: "스터디"(↔ 모임), "롤백"(↔ 되돌린다).
- 밀집 검색은 그런 질문에 강합니다. 이 예에서는 `E-4012` / `E-4021` 같은 코드도 구별했지만, 일반적으로 드문 식별자·제품 코드·숫자는 희소 검색이 더 안정적이어서 실제 서비스에서는 두 방법을 결합하는 구성이 흔합니다.
- 결합(RRF)이 항상 최고는 아닙니다. "스터디" 질문은 희소 검색의 나쁜 순위에 끌려 내려갔습니다.
- 질문 14개짜리 평가로는 **0.07(= 한 문항) 차이를 결론으로 삼을 수 없습니다.** → [Lab 03](lab03_llm_eval_error_bars.ipynb)에서 오차 막대를 다룹니다.

## 4. 검색한 문서를 넣어 답하게 하기 (RAG; Lewis et al., 2020)

RAG의 구조는 단순합니다: **검색(Retrieve) → 프롬프트에 붙이기(Augment) → 생성(Generate)**.  
작은 공개 모델(0.5B)로 직접 돌려 봅니다. 큰 모델보다 서툴지만 **모든 단계를 눈으로 볼 수 있습니다**.

In [6]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
llm_name = "Qwen/Qwen2.5-0.5B-Instruct"
tok = AutoTokenizer.from_pretrained(llm_name)
llm = AutoModelForCausalLM.from_pretrained(llm_name).eval()
print("파라미터 수:", round(sum(p.numel() for p in llm.parameters()) / 1e6), "M")

SYSTEM = "당신은 데이터팀의 사내 문서 도우미입니다. 한국어로 한두 문장으로 간결하게 답하세요."

def build_prompt(question, contexts=None):
    if not contexts: return question
    joined = "\n".join(f"[문서 {i + 1}] {c}" for i, c in enumerate(contexts))
    return f"다음 문서만 근거로 질문에 답하세요. 문서에 답이 없으면 '문서에 없습니다'라고 답하세요.\n\n{joined}\n\n질문: {question}"

def generate(question, contexts=None, max_new_tokens=80):
    messages = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": build_prompt(question, contexts)}]
    inputs = tok.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt", return_dict=True)
    with torch.no_grad():
        out = llm.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)      # do_sample=False: 매번 같은 답(greedy)
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip().replace("\n", " ")

def rag_answer(question, k=3):
    contexts = [docs[i] for i in rank_hybrid(question)[:k]]
    return generate(question, contexts)

q = queries[0][0]
print("모델에 실제로 들어가는 프롬프트:\n" + "-" * 60 + "\n" + build_prompt(q, [docs[i] for i in rank_hybrid(q)[:3]]) + "\n" + "-" * 60)
print("문서 없이 :", generate(q))
print("RAG      :", rag_answer(q))

파라미터 수: 494 M
모델에 실제로 들어가는 프롬프트:
------------------------------------------------------------
다음 문서만 근거로 질문에 답하세요. 문서에 답이 없으면 '문서에 없습니다'라고 답하세요.

[문서 1] 실험 정책: 우리 팀의 A/B 테스트 기본 유의수준은 0.01이며 최소 실험 기간은 14일이다. 요일 효과를 고려해 연장은 7일 단위로만 한다.
[문서 2] 배포 절차: 모델 배포는 매주 화요일 오전 10시에만 한다. 금요일 배포는 금지한다. 배포 전에 그림자(shadow) 테스트를 3일 동안 진행한다.
[문서 3] 데이터 분할 규칙: 모델 검증은 반드시 시간 기준으로 나눈다. 학습은 2025년 12월까지, 검증은 2026년 1월과 2월, 테스트는 2026년 3월 데이터를 쓴다.

질문: 실험은 최소 며칠 동안 돌려야 하나요?
------------------------------------------------------------


문서 없이 : 실험은 최소 1주일 동안 돌려야 합니다.


RAG      : 0.01 * 14 = 0.14 일


문서 없이 답할 때 모델은 **그럴듯한 답을 지어냅니다**(환각). 사내 규칙은 학습 데이터에 없으니 당연합니다.  
문서를 주면 답의 재료("14일")가 문맥에 들어옵니다. 다만 위 출력처럼 이 작은 모델은 그 재료를 엉뚱하게 조합하기도 합니다. RAG는 **근거를 공급**할 뿐, 근거를 올바르게 읽는 능력까지 주지는 않습니다. (실행 환경·라이브러리 버전에 따라 출력은 조금씩 달라질 수 있습니다.)

## 5. RAG를 평가하기: "문서 없이" vs "문서와 함께"

질문마다 **답에 꼭 들어가야 할 핵심 표현**(예: "14", "금지")을 정해 두었습니다. 포함 여부로 채점하는 가장 단순한 자동 평가입니다.

✍️ **예측**: 14문항 중 문서 없이 몇 개, RAG로 몇 개를 맞힐까요? 검색이 완벽한데도 RAG가 틀리는 경우가 있을까요?

In [7]:
rows = []
for question, gold, must_have in queries:
    retrieved = list(rank_hybrid(question)[:3])
    closed, rag = generate(question), generate(question, [docs[i] for i in retrieved])
    rows.append(dict(question=question[:24], retrieved_ok=gold in retrieved, closed_ok=must_have in closed, rag_ok=must_have in rag, rag_answer=rag[:70]))
report = pd.DataFrame(rows)
print(f"검색 성공(상위 3개에 정답 문서) = {report.retrieved_ok.mean():.2f}   문서 없이 정답 = {report.closed_ok.mean():.2f}   RAG 정답 = {report.rag_ok.mean():.2f}\n")
print(report[["question", "retrieved_ok", "closed_ok", "rag_ok", "rag_answer"]].to_string(index=False))

검색 성공(상위 3개에 정답 문서) = 0.93   문서 없이 정답 = 0.00   RAG 정답 = 0.64

                question  retrieved_ok  closed_ok  rag_ok                                                             rag_answer
   실험은 최소 며칠 동안 돌려야 하나요?          True      False    True                                                     0.01 * 14 = 0.14 일
       금요일에 모델을 올려도 되나요?          True      False   False           물론이죠! 금요일에 모델을 올릴 수 있습니다. 하지만 배포 전에 그림자 테스트를 3일 동안 진행해야 합니다.
     성능이 얼마나 나빠지면 롤백합니까?          True      False    True                                   AUC가 기준보다 0.03 이상 떨어지면 이전 버전으로 되돌린다.
      새 예측 모델을 채택하는 조건은?          True      False    True                                          새 모델은 MASE가 0.9 미만일 때만 채택합니다.
고객 정보를 언어모델에 보낼 때 지켜야 할           True      False    True 문서 1에서 언급된 내용은 개인정보를 마스킹하고 90일 동안 보관하는 것이며, 이는 고객 문의 요약에 적합합니다. 따라서 답변
문항이 200개인 시험으로 어느 모델이 더           True      False   False 문항이 200개인 시험에서는 [문서 2]에서의 데이터 분할 규칙을 따르므로, 모델 검증은 2025년 12월부터 시작되며, 검증
      테스트 세트는 언제 데이터인가요?          T

결과 읽는 법
- **검색 단계와 생성 단계를 따로 채점**했습니다. RAG가 틀렸을 때 "문서를 못 찾아서"인지 "찾았는데 잘못 읽어서"인지 구별하기 위해서입니다. 오류 분석의 첫걸음입니다.
- 문서 없이는 한 문항도 맞히지 못하고, RAG는 절반 이상을 맞힙니다. 검색은 거의 다 성공했는데 RAG 정답률이 그보다 낮다면 그 차이는 **생성 단계의 오류**입니다(0.5B 모델이 문서를 무시하거나, "금지"라고 적힌 문서를 보고도 "됩니다"라고 답하는 식).
- **채점기도 틀립니다.** 표의 첫 행을 보세요. 답이 말이 되지 않는데도 "14"가 들어 있어서 정답으로 처리됐습니다. "핵심 표현 포함 여부"는 가장 값싼 채점이지만 이렇게 거칠기 때문에, 실제로는 사람 검수나 LLM 채점(LLM-as-a-judge)을 쓰고 **그 채점기 자체도 사람 판단과 대조해 검증**합니다.

---
## 직접 해보기

1. **(k의 영향)** 프롬프트에 넣는 문서 수 k를 1, 3, 6으로 바꿔 RAG 정답률을 비교하세요. 문서를 많이 넣을수록 항상 좋아질까요?
2. **(모른다고 말하기)** 문서에 답이 없는 질문("우리 팀 회식은 언제인가요?")을 (a) 현재 프롬프트, (b) "문서에 답이 없으면…" 문장을 뺀 프롬프트로 물어보고 답을 비교하세요.
3. **(오류 분석)** 5절 표에서 틀린 문항을 하나 골라, 검색 문제인지 생성 문제인지 판단하고 고칠 방법을 한 줄로 적어 보세요.

In [8]:
# TODO 1: k = 1, 3, 6 일 때 RAG 정답률

# TODO 2: 문서에 없는 질문에 대한 답 비교

---
## 정답 예시 (먼저 풀어 본 뒤 확인)

In [9]:
# 1. k의 영향
for k in [1, 3, 6]:
    correct = [must in generate(q, [docs[i] for i in rank_hybrid(q)[:k]]) for q, _, must in queries]
    print(f"k={k}: RAG 정답률 = {np.mean(correct):.2f}")

k=1: RAG 정답률 = 0.57


k=3: RAG 정답률 = 0.64


k=6: RAG 정답률 = 0.71


In [10]:
# 2. 문서에 없는 질문
q = "우리 팀 회식은 언제인가요?"
ctx = [docs[i] for i in rank_hybrid(q)[:3]]
joined = "\n".join(f"[문서 {i + 1}] {c}" for i, c in enumerate(ctx))
no_guard = f"다음 문서를 참고해 질문에 답하세요.\n\n{joined}\n\n질문: {q}"
msgs = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": no_guard}]
inp = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt", return_dict=True)
with torch.no_grad(): out = llm.generate(**inp, max_new_tokens=60, do_sample=False)
print("(a) '없으면 없다고' 지시 있음:", generate(q, ctx))
print("(b) 지시 없음              :", tok.decode(out[0][inp["input_ids"].shape[1]:], skip_special_tokens=True).strip().replace("\n", " "))

(a) '없으면 없다고' 지시 있음: 문서 1에서 설명된 정보는 "A/B 테스트 기본 유의수준은 0.01이며 최소 실험 기간은 14일이다."라는 내용입니다. 따라서 회식은 14일에 이루어집니다.
(b) 지시 없음              : 그런 질문이 없네요. 저희 팀 회식은 언제나 주말에 진행됩니다.


**해석**
- 1번: k를 늘리면 정답 문서가 문맥에 들어올 가능성은 커지지만, 관련 없는 문서도 함께 늘어납니다. 이 실행의 차이는 한두 문항 수준이라(14문항) 결론을 내리기 어렵습니다. 큰 모델도 긴 문맥의 **가운데** 정보를 놓치는 경향이 보고되어 있습니다(*Lost in the Middle*). k는 충분한 문항으로 평가해서 정하는 하이퍼파라미터입니다.
- 2번: 이 0.5B 모델은 "문서에 없으면 없다고 답하라"는 지시가 있어도 관련 없는 문서에서 답을 지어냅니다. **지시를 따르는 능력 자체가 모델에 달려 있습니다.** 더 큰 모델에서는 이런 지시문이 효과가 있는 경우가 많지만, 그것도 가정하지 말고 "문서에 답이 없는 질문" 세트로 평가해 확인해야 합니다.

---
## 논문으로 돌아가기

| 방금 한 것 | 논문에서 확인할 곳 |
|---|---|
| 2절 | Sentence-BERT: 왜 BERT를 그대로 쓰면 문장 유사도 계산이 느린지, siamese 구조. DPR: 질문·문서 인코더를 따로 두고 in-batch negative로 학습 |
| 3절 | MTEB: 임베딩 모델을 과제 유형별(검색, 분류, 군집 …)로 평가하는 표. "우리 과제와 같은 유형의 점수"를 보는 법 |
| 4절 | RAG 논문의 구조 그림: retriever(DPR) + generator(BART), 그리고 검색기까지 **함께 학습**했다는 점(이 실습은 학습 없이 붙이기만 함) |
| 5절·해보기 1 | *Lost in the Middle* 의 U자 곡선 그림: 정답 문서의 위치에 따른 정확도 |

## 기억 점검

1. 희소 검색과 밀집 검색은 각각 어떤 질문에 강하고 약한가?
2. Recall@k와 MRR은 무엇을 재는가? 둘의 차이는?
3. RAG의 세 단계는? 답이 틀렸을 때 어느 단계의 문제인지 어떻게 구별하나?
4. 문서 없이 물으면 왜 환각이 생기고, RAG는 그것을 어떻게 줄이나? RAG로도 못 막는 오류는?
5. 문항 14개짜리 평가에서 정답률 0.71과 0.79의 차이를 어떻게 해석해야 하나?

<details><summary>답 확인</summary>

1. 희소: 정확한 용어·코드·숫자에 강하고 표현이 바뀌면 약함. 밀집: 바꿔 말한 질문에 강하고 드문 식별자·글자 하나 차이에 약할 수 있음.
2. Recall@k: 정답이 상위 k개 안에 있는가(있다/없다). MRR: 정답이 얼마나 위에 있는가(순위의 역수 평균).
3. 검색 → 프롬프트 구성 → 생성. 검색 결과에 정답 문서가 있었는지를 따로 기록해, 없으면 검색 문제, 있었는데 틀리면 생성(또는 프롬프트) 문제.
4. 모델 가중치에 없는 사실을 확률적으로 그럴듯하게 이어 쓰기 때문. RAG는 근거 문서를 문맥에 넣어 준다. 검색 실패, 문서를 잘못 읽는 생성 오류, 문서 자체의 오류는 여전히 남는다.
5. 한 문항 차이(1/14 ≈ 0.07)이므로 우연과 구별할 수 없다. 문항을 늘리고 신뢰구간을 계산해야 한다.
</details>